#Text Classification

**Course:** CS F429 Natural Language Processing  
**Topic:** TF-IDF, Bernoulli Naive Bayes, Multinomial Naive Bayes, Rocchio Classifier, k-NN, and Evaluation Metrics

This notebook solves the complete Tutorial IV problem step by step, using the same lecture-aligned method:

- first define the training and test documents,
- compute vocabulary, TF, DF, IDF and TF-IDF,
- normalize vectors,
- apply Bernoulli Naive Bayes,
- apply Multinomial Naive Bayes,
- apply Rocchio classifier,
- apply k-NN using cosine similarity,
- compute confusion matrix and evaluation metrics.

## Given Training Set

| Document | Text | Class |
|---|---|---|
| d1 | data data data mining algorithm | Tech |
| d2 | data algorithm model | Tech |
| d3 | football match goal team | Sports |
| d4 | match team tournament | Sports |

**Test document:**  
d5 = `data match algorithm`

In [1]:
import math
import itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

In [2]:
# Training documents
documents = {
    "d1": {"text": "data data data mining algorithm", "class": "Tech"},
    "d2": {"text": "data algorithm model", "class": "Tech"},
    "d3": {"text": "football match goal team", "class": "Sports"},
    "d4": {"text": "match team tournament", "class": "Sports"},
}

# Test document
test_document = "data match algorithm"

# Tokenization
for doc_id in documents:
    documents[doc_id]["tokens"] = documents[doc_id]["text"].split()

test_tokens = test_document.split()

documents

{'d1': {'text': 'data data data mining algorithm',
  'class': 'Tech',
  'tokens': ['data', 'data', 'data', 'mining', 'algorithm']},
 'd2': {'text': 'data algorithm model',
  'class': 'Tech',
  'tokens': ['data', 'algorithm', 'model']},
 'd3': {'text': 'football match goal team',
  'class': 'Sports',
  'tokens': ['football', 'match', 'goal', 'team']},
 'd4': {'text': 'match team tournament',
  'class': 'Sports',
  'tokens': ['match', 'team', 'tournament']}}

## 1. Vocabulary Construction

The vocabulary contains every unique word appearing in the training set.  
The test document is represented using this same vocabulary.

In [3]:
vocabulary = sorted(set(itertools.chain.from_iterable(
    doc["tokens"] for doc in documents.values()
)))

vocabulary

['algorithm',
 'data',
 'football',
 'goal',
 'match',
 'mining',
 'model',
 'team',
 'tournament']

## 2. Compute TF, DF, IDF and TF-IDF for d5

We use the lecture formula:

\[
tf_{t,d} = \log_{10}(count(t,d)+1)
\]

\[
idf_t = \log_{10}\left(\frac{N}{df_t}\right)
\]

\[
w_{t,d} = tf_{t,d} \times idf_t
\]

Here, \(N = 4\), because there are four training documents.

In [4]:
N = len(documents)

# Document frequency from training documents only
df = {
    term: sum(1 for doc in documents.values() if term in doc["tokens"])
    for term in vocabulary
}

idf = {
    term: math.log10(N / df[term])
    for term in vocabulary
}

# TF and TF-IDF for test document d5
test_counts = Counter(test_tokens)

tf_raw_d5 = {term: test_counts[term] for term in vocabulary}
tf_log_d5 = {
    term: math.log10(test_counts[term] + 1) if test_counts[term] > 0 else 0
    for term in vocabulary
}

tfidf_d5 = {
    term: tf_log_d5[term] * idf[term]
    for term in vocabulary
}

tfidf_table_d5 = pd.DataFrame({
    "Term": vocabulary,
    "Raw TF in d5": [tf_raw_d5[t] for t in vocabulary],
    "DF": [df[t] for t in vocabulary],
    "IDF = log10(4/DF)": [idf[t] for t in vocabulary],
    "Log TF = log10(TF+1)": [tf_log_d5[t] for t in vocabulary],
    "TF-IDF Weight": [tfidf_d5[t] for t in vocabulary],
})

tfidf_table_d5.round(4)

,Term,Raw TF in d5,DF,IDF = log10(4/DF),Log TF = log10(TF+1),TF-IDF Weight
0,algorithm,1,2,0.3010,0.301,0.0906
1,data,1,2,0.3010,0.301,0.0906
2,football,0,1,0.6021,0.000,0.0000
3,goal,0,1,0.6021,0.000,0.0000
4,match,1,2,0.3010,0.301,0.0906
5,mining,0,1,0.6021,0.000,0.0000
6,model,0,1,0.6021,0.000,0.0000
7,team,0,2,0.3010,0.000,0.0000
8,tournament,0,1,0.6021,0.000,0.0000


### Manual Interpretation for d5

Only three terms occur in d5:

- `data`
- `match`
- `algorithm`

Each occurs once, so:

\[
tf = \log_{10}(1+1)=0.3010
\]

For each of these terms, \(df = 2\), so:

\[
idf = \log_{10}(4/2)=0.3010
\]

Therefore:

\[
TFIDF = 0.3010 \times 0.3010 = 0.0906
\]

## 3. Length Normalization for d5

The length of the TF-IDF vector is:

\[
||d5|| = \sqrt{\sum_i w_i^2}
\]

Since d5 has three non-zero weights, each equal to 0.0906:

\[
||d5|| = \sqrt{0.0906^2 + 0.0906^2 + 0.0906^2}
\]

Then each non-zero normalized value becomes approximately:

\[
0.0906 / 0.157 = 0.577
\]

In [5]:
d5_vector = np.array([tfidf_d5[t] for t in vocabulary])
d5_length = np.linalg.norm(d5_vector)

d5_normalized = d5_vector / d5_length if d5_length != 0 else d5_vector

normalization_table = pd.DataFrame({
    "Term": vocabulary,
    "TF-IDF Weight": d5_vector,
    "Normalized Weight": d5_normalized
})

print("Length of d5 vector =", round(d5_length, 4))
normalization_table.round(4)

Length of d5 vector = 0.157


,Term,TF-IDF Weight,Normalized Weight
0,algorithm,0.0906,0.5774
1,data,0.0906,0.5774
2,football,0.0000,0.0000
3,goal,0.0000,0.0000
4,match,0.0906,0.5774
5,mining,0.0000,0.0000
6,model,0.0000,0.0000
7,team,0.0000,0.0000
8,tournament,0.0000,0.0000


## 4. Bernoulli Naive Bayes

Bernoulli Naive Bayes uses **binary representation**.

That means it checks whether a term is present or absent, not how many times it appears.

For each class:

\[
P(t|c)=\frac{\text{number of documents in class c containing t}+1}{N_c+2}
\]

The denominator uses \(+2\) because Bernoulli has two outcomes: present and absent.

For classification:

\[
P(c|d) \propto P(c) \prod_{t \in V} P(t|c)^{x_t}(1-P(t|c))^{1-x_t}
\]

where \(x_t = 1\) if term is present in d5, otherwise \(0\).

In [6]:
classes = sorted(set(doc["class"] for doc in documents.values()))

class_docs = {
    c: [doc_id for doc_id, doc in documents.items() if doc["class"] == c]
    for c in classes
}

priors = {
    c: len(class_docs[c]) / N
    for c in classes
}

bernoulli_likelihood = {}

for c in classes:
    Nc = len(class_docs[c])
    bernoulli_likelihood[c] = {}

    for term in vocabulary:
        docs_containing_term = sum(
            1 for doc_id in class_docs[c]
            if term in documents[doc_id]["tokens"]
        )
        bernoulli_likelihood[c][term] = (docs_containing_term + 1) / (Nc + 2)

bernoulli_table = pd.DataFrame({
    "Term": vocabulary,
    **{
        f"P({term_marker}|{c})": [bernoulli_likelihood[c][t] for t in vocabulary]
        for c in classes
        for term_marker in ["t"]
    }
})

bernoulli_table.round(4)

,Term,P(t|Sports),P(t|Tech)
0,algorithm,0.25,0.75
1,data,0.25,0.75
2,football,0.50,0.25
3,goal,0.50,0.25
4,match,0.75,0.25
5,mining,0.25,0.50
6,model,0.25,0.50
7,team,0.75,0.25
8,tournament,0.50,0.25


In [7]:
# Bernoulli posterior scores for d5
test_binary = {term: int(term in test_tokens) for term in vocabulary}

bernoulli_scores = {}

for c in classes:
    score = priors[c]
    steps = []

    for term in vocabulary:
        p = bernoulli_likelihood[c][term]
        if test_binary[term] == 1:
            score *= p
            steps.append((term, "present", p))
        else:
            score *= (1 - p)
            steps.append((term, "absent", 1 - p))

    bernoulli_scores[c] = score

bernoulli_result = pd.DataFrame({
    "Class": list(bernoulli_scores.keys()),
    "Prior": [priors[c] for c in bernoulli_scores],
    "Bernoulli Posterior Score": [bernoulli_scores[c] for c in bernoulli_scores],
})

bernoulli_result.round(8)

,Class,Prior,Bernoulli Posterior Score
0,Sports,0.5,0.000412
1,Tech,0.5,0.005562


In [8]:
bernoulli_prediction = max(bernoulli_scores, key=bernoulli_scores.get)
print("Bernoulli Naive Bayes Prediction for d5:", bernoulli_prediction)

Bernoulli Naive Bayes Prediction for d5: Tech


### Bernoulli NB Decision

Bernoulli NB predicts **Tech**, because the posterior score for Tech is greater than the posterior score for Sports.

Important observation: Bernoulli NB ignores that `data` appears three times in d1. It only considers whether `data` is present or absent.

## 5. Multinomial Naive Bayes

Multinomial Naive Bayes uses **frequency-based representation**.

For each class:

\[
P(t|c)=\frac{count(t,c)+1}{\sum_{t' \in V} count(t',c)+|V|}
\]

Here, \(|V|=9\).

For d5:

\[
P(c|d5) \propto P(c)P(data|c)P(match|c)P(algorithm|c)
\]

In [9]:
# Multinomial likelihoods
multinomial_likelihood = {}

for c in classes:
    all_tokens_in_class = []
    for doc_id in class_docs[c]:
        all_tokens_in_class.extend(documents[doc_id]["tokens"])

    class_token_counts = Counter(all_tokens_in_class)
    total_tokens_in_class = sum(class_token_counts.values())

    multinomial_likelihood[c] = {
        term: (class_token_counts[term] + 1) / (total_tokens_in_class + len(vocabulary))
        for term in vocabulary
    }

multinomial_table = pd.DataFrame({
    "Term": vocabulary,
    **{
        f"P(t|{c})": [multinomial_likelihood[c][t] for t in vocabulary]
        for c in classes
    }
})

multinomial_table.round(4)

,Term,P(t|Sports),P(t|Tech)
0,algorithm,0.0625,0.1765
1,data,0.0625,0.2941
2,football,0.1250,0.0588
3,goal,0.1250,0.0588
4,match,0.1875,0.0588
5,mining,0.0625,0.1176
6,model,0.0625,0.1176
7,team,0.1875,0.0588
8,tournament,0.1250,0.0588


In [10]:
# Multinomial posterior scores for d5
multinomial_scores = {}

for c in classes:
    score = priors[c]
    for term in test_tokens:
        score *= multinomial_likelihood[c][term]
    multinomial_scores[c] = score

multinomial_result = pd.DataFrame({
    "Class": list(multinomial_scores.keys()),
    "Prior": [priors[c] for c in multinomial_scores],
    "Multinomial Posterior Score": [multinomial_scores[c] for c in multinomial_scores],
})

multinomial_result.round(8)

,Class,Prior,Multinomial Posterior Score
0,Sports,0.5,0.000366
1,Tech,0.5,0.001527


In [11]:
multinomial_prediction = max(multinomial_scores, key=multinomial_scores.get)
print("Multinomial Naive Bayes Prediction for d5:", multinomial_prediction)

Multinomial Naive Bayes Prediction for d5: Tech


### Multinomial NB Decision

Multinomial NB predicts **Tech**.

This is because `data` and `algorithm` are much more associated with the Tech class, while `match` supports Sports. The combined probability is still larger for Tech.

## 6. Rocchio Classifier

Rocchio is a **geometric classifier**.

Steps:

1. Convert all documents into TF-IDF vectors.
2. Normalize document vectors.
3. Compute one centroid per class.
4. Normalize centroids.
5. Compute cosine similarity between d5 and each class centroid.
6. Assign d5 to the class with the highest cosine similarity.

In [12]:
def compute_tfidf_vector(tokens, vocabulary, idf):
    counts = Counter(tokens)
    vector = []
    for term in vocabulary:
        tf = math.log10(counts[term] + 1) if counts[term] > 0 else 0
        vector.append(tf * idf[term])
    return np.array(vector, dtype=float)

def normalize(vector):
    length = np.linalg.norm(vector)
    return vector / length if length != 0 else vector

# TF-IDF vectors for training documents
tfidf_vectors = {
    doc_id: compute_tfidf_vector(doc["tokens"], vocabulary, idf)
    for doc_id, doc in documents.items()
}

normalized_vectors = {
    doc_id: normalize(vector)
    for doc_id, vector in tfidf_vectors.items()
}

training_vector_table = pd.DataFrame(
    [normalized_vectors[doc_id] for doc_id in documents],
    index=documents.keys(),
    columns=vocabulary
)

training_vector_table.round(4)

,algorithm,data,football,goal,match,mining,model,team,tournament
d1,0.3333,0.6667,0.0000,0.0000,0.0000,0.6667,0.0000,0.0000,0.0000
d2,0.4082,0.4082,0.0000,0.0000,0.0000,0.0000,0.8165,0.0000,0.0000
d3,0.0000,0.0000,0.6325,0.6325,0.3162,0.0000,0.0000,0.3162,0.0000
d4,0.0000,0.0000,0.0000,0.0000,0.4082,0.0000,0.0000,0.4082,0.8165


In [13]:
# Compute class centroids from normalized training vectors
centroids = {}

for c in classes:
    vectors_in_class = [normalized_vectors[doc_id] for doc_id in class_docs[c]]
    centroids[c] = np.mean(vectors_in_class, axis=0)

normalized_centroids = {
    c: normalize(centroids[c])
    for c in classes
}

centroid_table = pd.DataFrame(
    [normalized_centroids[c] for c in classes],
    index=classes,
    columns=vocabulary
)

centroid_table.round(4)

,algorithm,data,football,goal,match,mining,model,team,tournament
Sports,0.0000,0.0000,0.3987,0.3987,0.4567,0.0000,0.0000,0.4567,0.5147
Tech,0.4419,0.6405,0.0000,0.0000,0.0000,0.3972,0.4865,0.0000,0.0000


In [14]:
# Cosine similarity between d5 and class centroids
d5_norm_vector = normalize(d5_vector)

rocchio_similarities = {
    c: float(np.dot(d5_norm_vector, normalized_centroids[c]))
    for c in classes
}

rocchio_result = pd.DataFrame({
    "Class": list(rocchio_similarities.keys()),
    "Cosine Similarity with d5": list(rocchio_similarities.values())
})

rocchio_result.round(4)

,Class,Cosine Similarity with d5
0,Sports,0.2637
1,Tech,0.6249


In [15]:
rocchio_prediction = max(rocchio_similarities, key=rocchio_similarities.get)
print("Rocchio Prediction for d5:", rocchio_prediction)

Rocchio Prediction for d5: Tech


### Rocchio Decision

Rocchio predicts **Tech**, because d5 has a higher cosine similarity with the Tech centroid than with the Sports centroid.

## 7. k-NN Classifier with k = 3

k-NN is also a geometric method.

Steps:

1. Represent d5 and all training documents as normalized TF-IDF vectors.
2. Compute cosine similarity between d5 and each training document.
3. Select the top k = 3 nearest neighbors.
4. Use majority voting.

In [16]:
# Cosine similarity between d5 and each training document
knn_similarities = {
    doc_id: float(np.dot(d5_norm_vector, normalized_vectors[doc_id]))
    for doc_id in documents
}

knn_table = pd.DataFrame({
    "Document": list(knn_similarities.keys()),
    "Class": [documents[doc_id]["class"] for doc_id in knn_similarities],
    "Cosine Similarity with d5": list(knn_similarities.values())
}).sort_values(by="Cosine Similarity with d5", ascending=False)

knn_table.round(4)

,Document,Class,Cosine Similarity with d5
0,d1,Tech,0.5774
1,d2,Tech,0.4714
3,d4,Sports,0.2357
2,d3,Sports,0.1826


In [17]:
k = 3
nearest_neighbors = knn_table.head(k)

vote_counts = nearest_neighbors["Class"].value_counts()
knn_prediction = vote_counts.idxmax()

print("Top 3 nearest neighbors:")
display(nearest_neighbors.round(4))

print("\nk-NN Prediction for d5:", knn_prediction)

Top 3 nearest neighbors:


,Document,Class,Cosine Similarity with d5
0,d1,Tech,0.5774
1,d2,Tech,0.4714
3,d4,Sports,0.2357



k-NN Prediction for d5: Tech


### k-NN Decision

The top three neighbors are:

1. d1 - Tech  
2. d2 - Tech  
3. d4 - Sports  

So the majority vote is **Tech**.

## 8. Critical Analysis

### Which model is most influenced by repetition?

**Multinomial Naive Bayes** is most influenced by repetition because it uses term frequencies.  
For example, `data` appears three times in d1, and this increases the likelihood of `data` under the Tech class.

### Which model ignores frequency?

**Bernoulli Naive Bayes** ignores frequency. It only checks whether a word is present or absent.

### Which methods are geometric?

**Rocchio** and **k-NN** are geometric methods because they use vector-space representation and similarity/distance measures.

### Do the models agree?

Yes. For d5, all four methods predict **Tech**:

| Model | Prediction |
|---|---|
| Bernoulli NB | Tech |
| Multinomial NB | Tech |
| Rocchio | Tech |
| k-NN | Tech |

The agreement is mathematically justified because d5 contains two Tech-indicative terms, `data` and `algorithm`, and one Sports-indicative term, `match`.

### Role of normalization in Rocchio and k-NN

Normalization ensures that cosine similarity compares the **direction** of document vectors rather than their raw length.  
Without normalization, longer documents or documents with repeated terms may dominate similarity scores unfairly.

## 9. Evaluation Metrics

Given test results:

| Test | Actual Class | Predicted Class |
|---|---|---|
| t1 | Tech | Tech |
| t2 | Sports | Sports |
| t3 | Tech | Sports |
| t4 | Sports | Sports |
| t5 | Tech | Tech |

Assume **Tech** is the positive class.

Then:

- TP = Tech predicted as Tech
- FP = Sports predicted as Tech
- TN = Sports predicted as Sports
- FN = Tech predicted as Sports

In [18]:
test_results = [
    ("t1", "Tech", "Tech"),
    ("t2", "Sports", "Sports"),
    ("t3", "Tech", "Sports"),
    ("t4", "Sports", "Sports"),
    ("t5", "Tech", "Tech"),
]

positive_class = "Tech"

TP = sum(1 for _, actual, predicted in test_results if actual == positive_class and predicted == positive_class)
FP = sum(1 for _, actual, predicted in test_results if actual != positive_class and predicted == positive_class)
TN = sum(1 for _, actual, predicted in test_results if actual != positive_class and predicted != positive_class)
FN = sum(1 for _, actual, predicted in test_results if actual == positive_class and predicted != positive_class)

confusion_matrix = pd.DataFrame(
    [[TP, FN],
     [FP, TN]],
    index=["Actual Tech", "Actual Sports"],
    columns=["Predicted Tech", "Predicted Sports"]
)

confusion_matrix

,Predicted Tech,Predicted Sports
Actual Tech,2,1
Actual Sports,0,2


In [19]:
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) != 0 else 0

metrics = pd.DataFrame({
    "Metric": ["TP", "FP", "TN", "FN", "Precision", "Recall", "Accuracy", "F1-score"],
    "Value": [TP, FP, TN, FN, precision, recall, accuracy, f1_score]
})

metrics

,Metric,Value
0,TP,2.000000
1,FP,0.000000
2,TN,2.000000
3,FN,1.000000
4,Precision,1.000000
5,Recall,0.666667
6,Accuracy,0.800000
7,F1-score,0.800000


### Final Evaluation Values

\[
TP = 2, \quad FP = 0, \quad TN = 2, \quad FN = 1
\]

\[
Precision = \frac{TP}{TP+FP} = \frac{2}{2+0} = 1.00
\]

\[
Recall = \frac{TP}{TP+FN} = \frac{2}{2+1} = 0.667
\]

\[
Accuracy = \frac{TP+TN}{TP+TN+FP+FN} = \frac{2+2}{5} = 0.80
\]

\[
F1 = \frac{2PR}{P+R} = \frac{2(1)(0.667)}{1+0.667} = 0.80
\]

## Final Summary

| Task | Output |
|---|---|
| TF-IDF for d5 | Non-zero weights for `data`, `match`, `algorithm` |
| Normalized d5 vector | Each non-zero value approximately 0.577 |
| Bernoulli NB | Tech |
| Multinomial NB | Tech |
| Rocchio | Tech |
| k-NN, k=3 | Tech |
| TP, FP, TN, FN | TP=2, FP=0, TN=2, FN=1 |
| Precision | 1.00 |
| Recall | 0.667 |
| Accuracy | 0.80 |
| F1-score | 0.80 |